# Solutions · Module 02 assessment

Mark Parts A and C against these. Part B marked itself; the working is here so you can find *where*
an answer went wrong rather than only that it did.

**Marking Part A:** 1 mark for each answer that contains the idea. Half marks are not worth the
argument - if you would accept it from a colleague, take the mark.

**Marking Part C:** the mark scheme is stated per question. C1 and C3 award most of their marks for
the *reasoning*, not the numbers, because the numbers are the easy half.

In [1]:
import hashlib

import numpy as np
import pandas as pd


def load_loans():
    """SYNTHETIC. One row per loan event at a two-branch city library."""
    rng = np.random.default_rng(2026)
    n_members = 120
    intensity = rng.gamma(2.0, 1.0, n_members) + 0.3
    loans_each = np.maximum(1, np.round(intensity * 4)).astype(int)
    member_id = np.repeat(np.arange(1, n_members + 1), loans_each)
    n = len(member_id)
    mu = np.clip(34 - 7.0 * intensity[member_id - 1], 5, None)
    days = np.clip(np.round(rng.gamma(4.0, mu / 4.0)), 1, None)
    days = np.minimum(days, 28)
    branch = rng.choice(["central", "riverside"], n, p=[0.45, 0.55])
    fmt = np.where(rng.random(n) < np.where(branch == "central", 0.75, 0.25), "print", "ebook")
    base = np.where(branch == "central", 0.62, 0.38)
    renewed = (rng.random(n) < base + np.where(fmt == "ebook", 0.10, 0.0)).astype(int)
    days = np.where(rng.random(n) < 0.08, -1, days)
    loans = pd.DataFrame({"loan_id": np.arange(1, n + 1), "member_id": member_id,
                          "branch": branch, "format": fmt,
                          "days_kept": days.astype(int), "renewed": renewed})
    return pd.concat([loans, loans.iloc[100:175].copy()], ignore_index=True)


EXPECTED = {
    "B1": "3b90cedb8dc1", "B2": "b3ba5ed83382", "B3": "3f408706c437", "B4": "13d621feea07",
    "B5": "c1a2fb15ff02", "B6": "b77f7f05789d", "B7": "333e2cde6851", "B8": "9563004c0f78",
}


def check(task, answer):
    """Marks one Part B answer without revealing it. Counts: whole numbers. Rates and means: 2 dp."""
    task = task.upper()
    if task not in EXPECTED:
        print("unknown task:", task)
        return
    candidates = [answer] if isinstance(answer, (int, np.integer)) else [
        round(float(answer) + delta, 2) for delta in (-0.01, 0.0, 0.01)
    ]
    for value in candidates:
        text = str(int(value)) if isinstance(answer, (int, np.integer)) else "%.2f" % value
        if hashlib.sha256((task + "|" + text).encode()).hexdigest()[:12] == EXPECTED[task]:
            print("%s  correct" % task)
            return
    print("%s  not yet - check your working, then try again" % task)


loans = load_loans()
print("loaded", loans.shape)

loaded (1176, 6)


## Part A · model answers

**A1.** A row is one **loan event**. A per-row average over-weights members who borrow a lot, so
anything phrased "the average member..." is wrong when computed per row - for example "how long does
a member keep a book", where heavy borrowers contribute many short loans each. *(02-01)*

**A2.** A re-imported batch, a retried upload, or a join that duplicated the left side. **Not always
the right fix**: identical rows can be genuinely distinct events (two copies of the same book
borrowed the same day) when there is no unique key. Here there *is* a key - `loan_id` - so
duplication of the key is a defect. *(02-02, 02-04)*

**A3.** (1) It is a valid number, so it survives every `isna()` check and every type check silently.
(2) It participates in arithmetic - means, sums, comparisons - and drags them in a direction nobody
chose. A third: filters like `days_kept < 7` silently select it. *(02-04)*

**A4.** Missing-at-random means whether a value is missing does not depend on the missing value
itself (once you account for what you observe); missing-not-at-random means it does - the reason for
missingness is related to the value. **Mean-imputation makes MNAR worse**: it pulls values towards a
centre they never belonged to and shrinks the variance, giving a confidently wrong answer. *(02-04)*

**A5.** **Count the rows at the maximum, and look at the next value down.** A genuine maximum is one
row, or a few, with the next value close behind. A ceiling is a crowd at the top and a cliff below
it. Suspicious roundness of the value itself is a hint, not evidence. *(02-05, 02-08)*

**A6.** Because the outlier inflates the standard deviation that the rule is built from - it moves
the goalposts. With one extreme value the three-sigma threshold can sit above every point, and
moderate outliers hide behind it. This is masking. *(02-05)*

**A7.** A comparison that goes one way within every subgroup and the other way when the subgroups are
pooled. **The condition: the groups are unevenly allocated across a factor that affects the outcome**
- so pooling compares populations rather than treatments. *(02-06)*

**A8.** A **causal** question. If the grouping variable is a **confounder** - it affects both the
comparison and the outcome - use the subgroups. If it is a **mediator** - it lies on the causal path
you care about - use the pooled total, since adjusting for it removes the effect you are measuring.
*(02-06, 00-04)*

**A9.** **The distribution of the best-of-30 correlation when nothing is related.** On 100 rows a
single correlation is typically around 0.07, and the best of thirty is typically around 0.24 - so
0.28 is barely above the noise benchmark and is not by itself a finding. Also: check it on data not
used to select it. *(02-07)*

**A10.** **Can settle:** what is in the data, what one row means, what is missing or duplicated,
what the distributions look like, whether a column has a ceiling, whether it was recorded
consistently. **Cannot settle:** the direction of a causal arrow; what happens if you intervene;
whether an outlier is an error or a real rare case; whether a relationship found by searching is
real; who is missing from the sample in a way you did not think to check. *(02-07)*

## Part B · worked

Order matters. De-duplicate first, then handle the sentinel, then look at the ceiling. Doing it in
another order gives numbers that are individually defensible and jointly inconsistent.

In [2]:
# B1, B2 - duplication
print("B1  exact duplicate rows      :", int(loans.duplicated().sum()))
distinct = loans.drop_duplicates("loan_id")
print("    delivered rows            :", len(loans))
print("    distinct loans            :", len(distinct))
print("B2  overstatement (percent)   : %.2f" % (100 * (len(loans) / len(distinct) - 1)))

B1  exact duplicate rows      : 75
    delivered rows            : 1176
    distinct loans            : 1101
B2  overstatement (percent)   : 6.81


In [3]:
# B3, B4, B5 - the sentinel
unreturned = distinct["days_kept"] == -1
returned = distinct.loc[~unreturned]

print("B3  loans not yet returned    :", int(unreturned.sum()),
      "  (%.2f%% of distinct loans)" % (100 * unreturned.mean()))
print("B4  naive mean days_kept      : %.2f   <- includes -1, so it is meaningless"
      % distinct["days_kept"].mean())
print("B5  mean days for returned    : %.2f" % returned["days_kept"].mean())
print()
print("    the sentinel drags the naive mean down by %.2f days"
      % (returned["days_kept"].mean() - distinct["days_kept"].mean()))

B3  loans not yet returned    : 86   (7.81% of distinct loans)
B4  naive mean days_kept      : 11.93   <- includes -1, so it is meaningless
B5  mean days for returned    : 13.03

    the sentinel drags the naive mean down by 1.10 days


In [4]:
# B6 - the ceiling
counts = returned["days_kept"].value_counts().sort_index()
top = returned["days_kept"].max()
print("B6  returned loans at the maximum (%d days): %d  (%.2f%%)"
      % (top, (returned["days_kept"] == top).sum(),
         100 * (returned["days_kept"] == top).mean()))
print()
print("    the five largest values and their counts:")
print(counts.tail(5).to_string())

B6  returned loans at the maximum (28 days): 114  (11.23%)

    the five largest values and their counts:
days_kept
24     15
25     14
26     22
27     12
28    114


**B6 is the one that tests whether you looked.** 114 loans sit at exactly 28 days and only 12 sit at
27 - a crowd and then a cliff. Twenty-eight days is a standard library loan period, so `days_kept`
is censored: those 114 loans were kept *at least* 28 days, and how much longer is unrecorded.

Consequence, and the reason it is worth 2 marks: **B5 is an underestimate** and cannot be fixed by
any amount of care with the sentinel.

In [5]:
# B7 - the aggregation question
per_member = returned.groupby("member_id")["days_kept"].mean()

print("B5  mean over loans   : %.2f" % returned["days_kept"].mean())
print("B7  mean over members : %.2f" % per_member.mean())
print("    difference        : %.2f days" % (per_member.mean() - returned["days_kept"].mean()))
print()
print("    loans per member: min %d  median %.0f  max %d"
      % (per_member.index.map(returned["member_id"].value_counts()).min(),
         np.median(returned["member_id"].value_counts()),
         per_member.index.map(returned["member_id"].value_counts()).max()))

B5  mean over loans   : 13.03
B7  mean over members : 16.65
    difference        : 3.62 days

    loans per member: min 1  median 7  max 35


In [6]:
# B8 - renewal rate by format
print("B8  ebook renewal rate : %.3f" % distinct.loc[distinct["format"] == "ebook", "renewed"].mean())
print("    print renewal rate : %.3f" % distinct.loc[distinct["format"] == "print", "renewed"].mean())

B8  ebook renewal rate : 0.471
    print renewal rate : 0.543


## Part C · model answers

### C1 · Which format gets renewed more (5 marks)

*1 mark for the numbers, 2 for choosing the branch-level answer, 2 for the condition under which the
pooled answer would be right.*

In [7]:
by_branch = distinct.pivot_table(index="branch", columns="format", values="renewed", aggfunc="mean")
overall = distinct.groupby("format")["renewed"].mean()
allocation = distinct.pivot_table(index="branch", columns="format", values="loan_id", aggfunc="count")

print("RENEWAL RATE WITHIN EACH BRANCH")
print(by_branch.round(3).to_string())
print("\nOVERALL")
print(overall.round(3).to_string())
print("\nHOW THE FORMATS ARE ALLOCATED")
print(allocation.to_string())
print("\nshare of ebook loans that are at riverside: %.3f"
      % (allocation.loc["riverside", "ebook"] / allocation["ebook"].sum()))

RENEWAL RATE WITHIN EACH BRANCH
format     ebook  print
branch                 
central    0.647  0.618
riverside  0.416  0.351

OVERALL
format
ebook    0.471
print    0.543

HOW THE FORMATS ARE ALLOCATED
format     ebook  print
branch                 
central      133    387
riverside    430    151

share of ebook loans that are at riverside: 0.764


**The numbers.** Ebooks are renewed more at central (**0.647** against 0.618) and more at riverside
(**0.416** against 0.351). Pooled, ebooks are renewed **less** (0.471 against 0.543). Textbook
Simpson's paradox.

**The answer to give the director: the branch-level one - ebooks renew more.** The reason is in the
allocation table: **76.4% of ebook loans are at riverside**, where renewal is lower for everything,
including print. The pooled comparison is mostly comparing riverside with central, and reports that
difference as a property of the format.

**What would have to be true for the pooled answer to be right.** The pooled number would be the
right one if branch were a **mediator** rather than a confounder - that is, if choosing to stock a
format *caused* the branch pattern, so that the branch effect were part of the format's effect. For
example: if ebooks were only offered at riverside *because* being an ebook makes a loan more likely
to be lent from the low-renewal branch, then routing through riverside is part of what an ebook does
and should be counted against it.

Here that is not credible - branch allocation is a stocking decision made before any loan happens,
so branch is a confounder and the subgroup answer is the one that survives.

**Full marks require the third part.** "Use the subgroups" is a rule; knowing when the rule inverts
is the understanding. *(02-06)*

### C2 · Three numbers for one question (4 marks)

*1 mark per difference explained, 2 for the choice and its defence.*

- **B4 (11.93) versus B5 (13.03).** The naive mean includes 86 loans coded `-1` for "not returned".
  Those are not short loans, they are non-events, and averaging them in pulls the answer down by 1.10
  days. B4 is not a worse estimate of the same thing - it is an estimate of nothing.
- **B5 (13.03) versus B7 (16.65).** B5 averages over *loans*, B7 over *members*. They differ by 3.62
  days because members do not borrow equally: heavy borrowers keep books for less time and contribute
  many rows each, so the per-loan average is dominated by them.

**Which to report under the title "How long do our members keep books?"** - **B7, 16.65 days**,
because the title asks about members and B7 is the only one whose unit is a member. B5 answers "how
long is the typical loan kept", which is the right number for a shelf-availability question and the
wrong one for a member-behaviour question.

**And say the third thing:** both are underestimates, because 114 returned loans are censored at the
28-day cap and 86 loans have not been returned at all - and those 86 are disproportionately the long
ones, since a loan still out is a loan being kept a long time. A defensible report says *"at least 16.7
days, and the true figure is higher for reasons we can name."* *(02-01, 02-04, 02-05)*

### C3 · The honest finding (5 marks)

*1 mark per line, 1 for the intervention sentence.*

1. **Claim.** Within each branch, ebook loans **are renewed more often than** print loans.
2. **Evidence.** Central 0.647 against 0.618 (520 loans); riverside 0.416 against 0.351 (581 loans).
   Pooled across branches the comparison reverses to 0.471 against 0.543, because 76% of ebook loans
   are at riverside, where renewal is lower for both formats.
3. **What else could produce this.** Members who choose ebooks may differ from members who choose
   print in ways that affect renewal - they may be the heavier borrowers, or the ones with longer
   commutes. Renewing an ebook may simply be easier: if it is one tap in an app and a print renewal
   needs a visit or a phone call, the gap measures friction, not preference. Neither is testable from
   these columns.
4. **What would change my mind.** If the two branches stocked formats evenly next quarter and the
   within-branch gap disappeared, the finding was an artefact of stocking. If renewal friction were
   equalised - same one-tap renewal for both - and the gap closed, the finding was about the renewal
   mechanism rather than the format.

**Can this dataset tell the director what would happen if the library bought more ebooks?** **No.**
It shows an association measured on loans that members chose for themselves, and buying more ebooks
is an intervention - to answer it you would need to change the stock and watch, ideally varying it
across branches so the change is separable from everything else. *(02-06, 02-07)*

## After marking

Fill in the score table in the assessment notebook and use the remediation map there.

Two things worth noticing about your own answers, whatever the total:

- **Did you check the order of operations in Part B?** The commonest way to lose marks there is not
  an arithmetic slip but computing B5 before de-duplicating, or B6 without excluding the sentinel.
  Every one of those produces a plausible number.
- **In Part C, did you write what would change your mind?** It is the line people skip, and the one
  that separates a finding from an opinion.

**Next: 03-01.** Module 03 introduces the mathematics - notation first, then the small number of
ideas from linear algebra, calculus, probability and statistics the rest of the course actually
uses, each where it is used.